# RG3 F1: 배럴 온도 프로파일 모양 전문가

온도1~5의 공통 수준과 구분되는 선형/이차/삼차/사차 직교 대조4개 + 배럴-Hopper gap + 배럴-금형 gap + 금형 차이 + slope×curvature =8개. 배럴 센서 번호의 실제 물리 배치/활성 여부를 확정한 것은 아니며 수학적 프로파일 표현이다. 가스 원인 유형 확정/연소·건조 진단이라고 주장하지 않음. RF/ET 4모델 및 부스팅3모델 평균 전문가를 각각 기존 전문가에 추가.

RG3 RH620(정상601/가스19), 같은7개 관측구간. 94번 캐시 학습ref를 준수하고 코호트/그룹/시각 일치 assert. 모든 외부 평가 구간은 전처리·분류·가중치·임계값 학습에서 제외. 주결과는 inner_F1_selected_thermal_blend. 같은 개발 자료 구간검증이며 새 독립 검증 아님. 좋은 후보 행을 주결과로 바꾸지 않음.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
assert len(d)==620 and y.sum()==19 and np.array_equal(y,z['labels']) and np.array_equal(groups.to_numpy(),z['groups']) and np.array_equal(d.TimeStamp.astype(str),z['timestamps'])
old={}
for key in z.files:
 if key.startswith('ref_'):old[tuple(z[key].astype(int))]=z['scores_'+key.split('_')[1]]
b=d[[f'Barrel_Temperature_{j}' for j in range(1,6)]].to_numpy(float)
weights_profile=np.array([[-2,-1,0,1,2],[2,-1,-2,-1,2],[-1,2,0,-2,1],[1,-4,6,-4,1]],float)
weights_profile/=np.sum(weights_profile**2,axis=1)[:,None]
components=b@weights_profile.T
mean=np.mean(b,axis=1);mold=(d.Mold_Temperature_3+d.Mold_Temperature_4)/2
F=pd.DataFrame(components,columns=['profile_slope','profile_quadratic','profile_cubic','profile_quartic'])
F['barrel_hopper_gap']=mean-d.Hopper_Temperature
F['barrel_mold_gap']=mean-mold
F['mold_side_gap']=d.Mold_Temperature_3-d.Mold_Temperature_4
F['slope_curvature_interaction']=F.profile_slope*F.profile_quadratic
F=F.replace([np.inf,-np.inf],np.nan)
F.assign(TimeStamp=d.TimeStamp).to_csv(t/'rg3_thermal_shape_features.csv',index=False,encoding='utf-8-sig')
memo={};audit=[]
def predict(ref,q):
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 key=tuple(ref);assert key in old
 if key not in memo:
  normal=ref[y[ref]==0]
  im=SimpleImputer(strategy='median',keep_empty_features=True).fit(F.iloc[normal]);all_x=im.transform(F)
  forest=[];boost=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
    forest.append(m.fit(all_x[ref],y[ref]).predict_proba(all_x)[:,1])
  models=[
   CatBoostClassifier(iterations=150,depth=3,learning_rate=.05,l2_leaf_reg=10,auto_class_weights='Balanced',random_seed=42,thread_count=2,verbose=False,allow_writing_files=False),
   LGBMClassifier(n_estimators=150,num_leaves=7,max_depth=3,min_child_samples=15,learning_rate=.03,reg_lambda=10,class_weight='balanced',random_state=42,n_jobs=2,verbosity=-1),
   XGBClassifier(n_estimators=150,max_depth=2,min_child_weight=5,learning_rate=.03,reg_lambda=10,subsample=.8,colsample_bytree=.8,scale_pos_weight=float((y[ref]==0).sum()/y[ref].sum()),random_state=42,n_jobs=2,eval_metric='logloss')]
  for m in models:boost.append(m.fit(all_x[ref],y[ref]).predict_proba(all_x)[:,1])
  memo[key]=np.column_stack([old[key][:,[0,1,2,4,5]],np.mean(forest,axis=0),np.mean(boost,axis=0)])
  audit.append(dict(train=len(ref),gas_training=int(y[ref].sum()),train_blocks=','.join(map(str,groups.iloc[ref].unique())),excluded_blocks=','.join(map(str,groups[~groups.index.isin(ref)].unique()))))
 return memo[key][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,score):
 p,r,c=precision_recall_curve(yy,score)
 f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 order=np.lexsort((c,r[:-1],f));cut=float(c[order[-1]])
 return cut,stats(yy,score>=cut)
mixes=[np.eye(7)[i] for i in range(7)]+[(np.eye(7)[i]+np.eye(7)[j])/2 for i in range(7) for j in range(i+1,7)]
records=[];choices=[];search=[]
for block in blocks:
 q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block))
 out=predict(ref,q);inner=np.full((len(ref),7),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for w in mixes:
  cut,result=cutfit(y[ref],inner@w);score=out@w
  candidates.append((result['F1'],result['recall'],-result['FP'],w,cut,score))
  search.append(dict(block=int(block),weights=','.join(map(str,w)),threshold=cut,**result))
 best=max(candidates,key=lambda a:a[:3])
 procedures=[('inner_F1_selected_thermal_blend',best[5],best[4])]
 for col,name in [(5,'thermal_shape_forest'),(6,'thermal_shape_boost')]:
  cut,_=cutfit(y[ref],inner[:,col]);procedures.append((name+'_inner_F1_cut',out[:,col],cut))
  cut=float(np.quantile(inner[y[ref]==0,col],.99,method='higher'));procedures.append((name+'_normal99_cut',out[:,col],cut))
 for name,score,cut in procedures:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=d.TimeStamp.iloc[i],block=int(block),actual=int(y[i]),prediction=int(score[j]>=cut if name.endswith('F1_cut') or name.startswith('inner_') else score[j]>cut),score=float(score[j]),threshold=cut))
 choices.append(dict(block=int(block),weights=','.join(map(str,best[3])),threshold=best[4],inner_F1=best[0],inner_recall=best[1]))
 (t/'rg3_thermal_shape_progress.json').write_text(json.dumps(dict(last_completed_block=int(block),completed_training_refs=len(memo),total_blocks=len(blocks)),indent=2),encoding='utf-8')
 print('Finished RG3 thermal-shape block',block,flush=True)
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(search),'inner_search'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(t/f'rg3_thermal_shape_{name}.csv',index=False,encoding='utf-8-sig')
cache={'labels':y,'groups':groups.to_numpy(),'timestamps':d.TimeStamp.astype(str).to_numpy(dtype='U')}
for j,(ref,score) in enumerate(memo.items()):cache[f'ref_{j}']=np.array(ref,int);cache[f'scores_{j}']=score
np.savez_compressed(t/'rg3_thermal_shape_fold_cache.npz',**cache)
print(m.to_string(index=False))


Finished RG3 thermal-shape block 1


Finished RG3 thermal-shape block 2


Finished RG3 thermal-shape block 3


Finished RG3 thermal-shape block 4


Finished RG3 thermal-shape block 5


Finished RG3 thermal-shape block 6


Finished RG3 thermal-shape block 7


                            model       AP  TP  FP  FN  TN       F1   recall
  inner_F1_selected_thermal_blend 0.069945   3  19  16 582 0.146341 0.157895
 thermal_shape_boost_inner_F1_cut 0.021583  19 598   0   3 0.059748 1.000000
 thermal_shape_boost_normal99_cut 0.021583   0   2  19 599 0.000000 0.000000
thermal_shape_forest_inner_F1_cut 0.023634  19 599   0   2 0.059655 1.000000
thermal_shape_forest_normal99_cut 0.023634   0   3  19 598 0.000000 0.000000
